# Chapter 13: Deploying scikit-learn Models in Production

**Book:** *scikit-learn Cookbook, Third Edition* – John Sukup  
**Chapter summary and practical examples**

This notebook summarizes deployment, persistence, performance, monitoring, and model life-cycle practices using local, reproducible examples.

## Learning Objectives

- Describe the steps involved in moving a trained estimator into production.
- Save and reload models with `joblib` and understand serialization risks.
- Consider latency, throughput, batching, and parallel execution.
- Monitor model quality as new data arrives and respond to degradation.
- Track model versions, metadata, validation results, and dependencies.
- Package preprocessing and prediction steps into a repeatable pipeline.

## Environment Setup

A production model is more than a fitted estimator: it is an artifact that must be reproducible, loadable, testable, and monitored. The examples simulate deployment locally rather than creating a public service or cloud deployment.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
from joblib import dump, load
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 2024
ARTIFACT_DIR = Path("model_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)


## 13.1 Overview of Model Deployment

Deployment moves a trained model from development into an environment where real applications can request predictions. A typical workflow trains and validates the estimator, packages the model and supporting metadata, loads it in a serving process, tests predictions, and measures latency and throughput.

Models may be served through an API, used in scheduled batch jobs, or embedded into another application. Batch predictions can be more efficient than processing every record individually because they reduce per-request overhead.

In [2]:
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=12,
    random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
print(f"Validation accuracy: {accuracy_score(y_test, model.predict(X_test)):.3f}")


Validation accuracy: 0.660


## 13.2 Serialization and Persistence Techniques

Serialization converts a fitted estimator into a representation that can be stored and restored later. The chapter discusses Python's `pickle`, scikit-learn workflows using `joblib`, and exchange formats such as ONNX or PMML where supported.

`joblib` is convenient for scikit-learn estimators and NumPy-heavy objects. **Never load serialized artifacts from an untrusted source**, because Python object deserialization can execute malicious code. Models may also be incompatible across library versions, so record the environment used to create them.

In [3]:
model_path = ARTIFACT_DIR / "logistic_model.joblib"
dump(model, model_path)

loaded_model = load(model_path)
original_predictions = model.predict(X_test[:10])
loaded_predictions = loaded_model.predict(X_test[:10])

assert np.array_equal(original_predictions, loaded_predictions)
print("Loaded model matches the original predictions.")
print("Artifact:", model_path)


Loaded model matches the original predictions.
Artifact: model_artifacts\logistic_model.joblib


### Record Model Metadata

Persist useful context alongside the model: model name/version, training date, feature expectations, data or code revision, package versions, and validation metrics. Metadata helps teams reproduce results and determine which artifact is deployed. Avoid storing secrets or unnecessary personal information in metadata.

In [4]:
metadata = {
    "model_name": "logistic-classifier",
    "model_version": "1.0.0",
    "random_state": RANDOM_STATE,
    "scikit_learn_version": sklearn.__version__,
    "n_features": int(X.shape[1]),
    "validation_accuracy": float(accuracy_score(y_test, model.predict(X_test))),
    "artifact_file": model_path.name,
}
metadata_path = ARTIFACT_DIR / "metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))
print(metadata_path.read_text())


{
  "model_name": "logistic-classifier",
  "model_version": "1.0.0",
  "random_state": 2024,
  "scikit_learn_version": "1.9.1",
  "n_features": 20,
  "validation_accuracy": 0.66,
  "artifact_file": "logistic_model.joblib"
}


## 13.3 Scaling Models for Production

Scaling is about meeting demand while keeping response times acceptable. Relevant techniques include estimator-level parallelism (where supported), efficient batch prediction, and distributed computation tools such as Dask for larger workloads. The best choice depends on the estimator, data volume, hardware, and request pattern.

The `n_jobs` parameter commonly controls parallel work in supported estimators. More workers are not guaranteed to improve every workload, so benchmark with realistic data and measure both latency and throughput.

In [5]:
rf = RandomForestClassifier(
    n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)

batch = X_test
start = time.perf_counter()
batch_predictions = rf.predict(batch)
batch_seconds = time.perf_counter() - start

start = time.perf_counter()
single_predictions = np.array([rf.predict(row.reshape(1, -1))[0] for row in batch])
single_seconds = time.perf_counter() - start

print(f"Batch prediction: {batch_seconds:.6f} seconds for {len(batch)} rows")
print(f"One-row-at-a-time loop: {single_seconds:.6f} seconds for {len(batch)} rows")
print("Predictions equivalent:", np.array_equal(batch_predictions, single_predictions))


Batch prediction: 0.028846 seconds for 250 rows
One-row-at-a-time loop: 7.443256 seconds for 250 rows
Predictions equivalent: True


## 13.4 Monitoring and Updating Deployed Models

A model can become less effective when input patterns, user behavior, or the target process changes. This is often discussed as **data drift** or **concept drift**. Monitoring should track the signals that matter for the application: prediction latency, error rates, input distributions, and—when labels become available—predictive quality.

Some estimators, including `SGDClassifier`, support `partial_fit()` so they can be updated in stages. A simple performance threshold can trigger a review or retraining workflow, but real systems should also validate new models before replacing the currently deployed version.

In [6]:
X_stream, y_stream = make_classification(
    n_samples=400, n_features=10, n_informative=7,
    random_state=2025
)
X_initial, X_future = X_stream[:200], X_stream[200:]
y_initial, y_future = y_stream[:200], y_stream[200:]

online_model = SGDClassifier(loss="log_loss", random_state=RANDOM_STATE)
online_model.partial_fit(X_initial, y_initial, classes=np.array([0, 1]))

X_batches = np.array_split(X_future, 4)
y_batches = np.array_split(y_future, 4)
batch_accuracy = []

for X_batch, y_batch in zip(X_batches, y_batches):
    predictions = online_model.predict(X_batch)
    batch_accuracy.append(accuracy_score(y_batch, predictions))

print("Accuracy by incoming batch:", [round(x, 3) for x in batch_accuracy])
threshold = 0.70
if min(batch_accuracy) < threshold:
    print("Performance threshold crossed: review data and consider validated retraining.")
else:
    print("No batch crossed the illustrative threshold.")


Accuracy by incoming batch: [0.52, 0.42, 0.52, 0.66]
Performance threshold crossed: review data and consider validated retraining.


## 13.5 Managing the Model Life Cycle

A model life cycle includes training, validation, release, monitoring, updates, and eventual retirement. Reproducibility improves when teams version model artifacts and code, retain the relevant configuration, record validation results, and test that the serving environment produces expected outputs.

Do not assume a model is “set and forget.” Define how performance is reviewed, when retraining is considered, how a new version is approved, and how to roll back if the replacement behaves poorly.

In [7]:
# A compact release record that can be stored with the model artifact.
release_record = {
    "model_version": "1.0.0",
    "status": "validated-example",
    "validation_metric": "accuracy",
    "validation_score": float(accuracy_score(y_test, model.predict(X_test))),
    "sklearn_version": sklearn.__version__,
    "notes": "Synthetic demonstration; validate on representative production data before deployment."
}
release_path = ARTIFACT_DIR / "release_record.json"
release_path.write_text(json.dumps(release_record, indent=2))
print("Release record saved to:", release_path)


Release record saved to: model_artifacts\release_record.json


## 13.6 Setting Up Deployment Pipelines

A deployment pipeline makes the path from data to a released model repeatable. A practical sequence is:

1. Validate incoming training data and feature definitions.
2. Train candidate models and evaluate them with the agreed metrics.
3. Save the approved estimator and its metadata.
4. Run consistency checks after reloading the artifact.
5. Package the artifact with compatible dependencies.
6. Deploy to the target serving environment and check health and latency.
7. Monitor performance and input changes; retrain and release a new version only after validation.

A scikit-learn `Pipeline` is useful because it keeps preprocessing and estimation together, reducing the risk that training and prediction use different transformations.

In [8]:
pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
pipeline.fit(X_train, y_train)

pipeline_path = ARTIFACT_DIR / "complete_pipeline.joblib"
dump(pipeline, pipeline_path)
reloaded_pipeline = load(pipeline_path)

pipeline_predictions = reloaded_pipeline.predict(X_test)
print(f"Reloaded pipeline accuracy: {accuracy_score(y_test, pipeline_predictions):.3f}")
assert np.array_equal(pipeline.predict(X_test[:10]), reloaded_pipeline.predict(X_test[:10]))
print("Pipeline reload check passed.")


Reloaded pipeline accuracy: 0.660
Pipeline reload check passed.


## Chapter 13 Summary

Production deployment requires more than fitting an estimator. Models need reliable persistence, compatible dependencies, efficient serving, monitoring, version control, and a defined update process. `joblib` is a practical option for saving scikit-learn models, but serialized artifacts must be trusted and versioned. Batch inference and parallel execution may improve throughput, while monitoring helps reveal changes that warrant investigation or retraining. A reproducible pipeline connects preprocessing and prediction into one deployable artifact.